In [ ]:
import random

import minitorch
import minitorch.nn.functional as F
from minitorch import nn, optim
from minitorch.utils.data import DataLoader, Dataset

In [ ]:
random.seed(42)
minitorch.manual_seed(42)

MAX_NUM        = 9999
NUM_SAMPLES    = 20_000
VAL_SAMPLES    = 1_000
BLOCK_SIZE     = len(str(MAX_NUM)) * 3 + 3

VOCAB          = list("0123456789+=") + ["<eos>"]
CHAR_TO_IDX    = {c: i for i, c in enumerate(VOCAB)}
IDX_TO_CHAR    = {i: c for c, i in CHAR_TO_IDX.items()}
VOCAB_SIZE     = len(VOCAB)
EOS_IDX        = CHAR_TO_IDX["<eos>"]

N_EMBED        = 64
N_HEADS        = 4
N_LAYER        = 2
DROPOUT        = 0.1
BATCH_SIZE     = 256
EPOCHS         = 5
LR             = 1e-3

In [ ]:
def encode(s: str) -> list[int]:
    return [CHAR_TO_IDX[c] for c in s]


def decode(tokens: list[int]):
    return "".join([IDX_TO_CHAR[i] for i in tokens])


class AdditionDataset(Dataset):
    def __init__(self, num_samples: int, max_num: int = MAX_NUM):
        self.width = len(str(max_num))
        self.data = [
            self.make(random.randint(0, max_num), random.randint(0, max_num))
            for _ in range(num_samples)
        ]

    def make(self, a: int, b: int):
        w: int = self.width
        prompt = f"{a:0{w}d}+{b:0{w}d}="
        # reversed so that it follows how human calculate add
        answer = f"{a + b:0{w + 1}d}"[::-1]
        ids = encode(prompt +  answer) + [EOS_IDX]
        x = minitorch.tensor(ids[:-1], dtype=minitorch.long)
        y = minitorch.tensor(ids[1:], dtype=minitorch.long)
        # no loss on the prompt
        y[: len(prompt) - 1] = -100
        return x, y

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        return self.data[index]

In [ ]:
class CausalSingleHeadSelfAttention(nn.Module):
    tril: minitorch.Tensor

    def __init__(self, n_embed, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.register_buffer('tril', minitorch.tril(minitorch.ones(BLOCK_SIZE, BLOCK_SIZE)))

    def forward(self, x):
        _, T, _ = x.shape
        q: minitorch.Tensor = self.query(x) # (B, T, C)
        k: minitorch.Tensor = self.key(x) # (B, T, C)
        wei: minitorch.Tensor = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5 # (B, Tq, Tk)
        wei = wei.masked_fill( self.tril[:T, :T] == 0, float("-inf"))
        wei = wei.softmax(-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v # (B, T, C)
        return out

class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.heads = nn.ModuleList(
            [CausalSingleHeadSelfAttention(n_embed, head_size) for _ in range(n_heads)]
        )
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = minitorch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class MLP(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, n_embed*2),
            nn.ReLU(),
            nn.Linear(n_embed*2, n_embed),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        self.attn = CausalMultiHeadSelfAttention(n_embed, n_heads)
        self.mlp = MLP(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(VOCAB_SIZE, N_EMBED)
        self.wpe = nn.Embedding(BLOCK_SIZE, N_EMBED)
        self.drop = nn.Dropout(DROPOUT)
        self.h = nn.ModuleList([TransformerBlock(N_EMBED, N_HEADS) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBED)
        self.lm_head = nn.Linear(N_EMBED, VOCAB_SIZE, bias=False)

    def forward(self, context: minitorch.Tensor):
        b, t = context.size()
        assert t <= BLOCK_SIZE
        pos = minitorch.arange(0, t)
        tok_ebd = self.wte(context)
        pos_ebd = self.wpe(pos)
        x = self.drop(tok_ebd + pos_ebd)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
train_ds = AdditionDataset(NUM_SAMPLES)
val_ds   = AdditionDataset(VAL_SAMPLES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = Transformer()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


def train_epoch():
    model.train()
    total_loss = 0.0
    for context, targets in train_loader:
        logits = model(context)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@minitorch.no_grad()
def evaluate():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for context, targets in val_loader:
        logits = model(context)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        total_loss += loss.item()
        preds = logits.argmax(dim=-1) # (B, T)
        match = (preds == targets) | (targets == -100)
        correct += match.view(B, -1).all(dim=1).sum().item()
        total   += B
    return total_loss / len(val_loader), correct / total

@minitorch.no_grad()
def predict(a: int, b: int) -> int:
    model.eval()
    w = len(str(MAX_NUM))
    context = minitorch.tensor([encode(f"{a:0{w}d}+{b:0{w}d}=")], dtype=minitorch.long)
    chars = []
    for _ in range(w + 1):
        logits = model(context)
        next_token = logits[0, -1].argmax().item()
        next_tok = minitorch.tensor([[next_token]], dtype=minitorch.long)
        context = minitorch.cat((context, next_tok), dim=1)
        if next_token == EOS_IDX:
            break
        chars.append(decode([next_token]))
    # reverse back
    chars.reverse()
    result_str = "".join(chars).lstrip("0") or "0"
    return int(result_str)

In [ ]:
history = {
    "train_loss": [], "val_loss": [], "val_acc": [],
}
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    val_loss, val_acc = evaluate()
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  {epoch:>2}/{EPOCHS:<5} {train_loss:>7.4f} {val_loss:>7.4f} {val_acc*100:>6.1f}%")

print("=" * 95)

In [ ]:
predict(42, 4)